# Manual minimizer parameter update

Load a GUNDAM engine, modify one minimizer fit parameter by hand, propagate the model, and evaluate the likelihood.

In [1]:
nCpuThreads = 3
gundamLibPath = "/Users/nadrino/Documents/Work/Install/gundam/lib"
workDir = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024"
configPath = "configOA2024.yaml"
overrideList = [
    "override/v12ProdRun45.yaml",
    "override/onlyFlux5.yaml",
    "override/noEigen.yaml",
]
dataType = "Toy"  # "Asimov", "Toy", or "RealData"
seed = 12345

In [2]:
import sys
from pathlib import Path

import numpy as np

repoRoot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
srcPath = repoRoot / "src"
if srcPath.exists() and str(srcPath) not in sys.path:
    sys.path.insert(0, str(srcPath))

from gundam_interface import GundamInterface, GundamLoader, GundamRuntime

In [3]:
np.random.seed(seed)

runtime = GundamRuntime(
    loader=GundamLoader(gundamLibPath=gundamLibPath),
    workDir=workDir,
    nCpuThreads=nCpuThreads,
    configPath=configPath,
    overrideList=overrideList,
    dataType=dataType,
    randomSeed=seed,
)

runtime.toDict(includeConfigJsonString=False)

{'nCpuThreads': 3,
 'workDir': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024',
 'dataType': 'Toy',
 'loader': {'moduleName': 'GUNDAM',
  'gundamLibPath': '/Users/nadrino/Documents/Work/Install/gundam/lib'},
 'randomSeed': 12345,
 'configPath': 'configOA2024.yaml',
 'overrideList': ['override/v12ProdRun45.yaml',
  'override/onlyFlux5.yaml',
  'override/noEigen.yaml']}

In [4]:
gundam = GundamInterface(runtime)
gundam.configure()
gundam.initialize()

2026.06.30 17:21:51  INFO ConfigUtils: Reading config file: /Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/configOA2024.yaml
2026.06.30 17:21:51  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/v12ProdRun45.yaml"
2026.06.30 17:21:51  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/mc/filePathList
2026.06.30 17:21:51  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/data/0(name:"data")/filePathList
2026.06.30 17:21:51  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/onlyFlux5.yaml"
2026.06.30 17:21:51  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSetListConfig/0(name:"Flux Systematics")/isEnabled: true -> false
2026.06.30 17:21:51  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSe

In [5]:
fitParameters = gundam.minimizerFitParameters

for index, parameter in enumerate(fitParameters):
    print(
        f"{index:3d}: {parameter.getFullTitle()} "
        f"value={parameter.getParameterValue():.8g}"
    )

  0: Flux Systematics/#0 value=1
  1: Flux Systematics/#1 value=1
  2: Flux Systematics/#2 value=1
  3: Flux Systematics/#3 value=1
  4: Flux Systematics/#4 value=1


In [6]:
likelihoodInterface = gundam.engine.getLikelihoodInterface()
likelihoodInterface.propagateAndEvalLikelihood()
nominalLlh = float(likelihoodInterface.getLastLikelihood())

print(f"Nominal LLH: {nominalLlh:.8g}")

Nominal LLH: 4033.475


In [7]:
parameterIndex = 0
delta = 0.1

parameter = fitParameters[parameterIndex]
initialValue = float(parameter.getParameterValue())
print(f"Initial value: {float(parameter.getParameterValue()):.8g}")

updatedValue = initialValue + delta

print(f"Updated value: {float(updatedValue):.8g}")
parameter.setParameterValue(updatedValue)

print(parameter.getFullTitle())
print(f"Initial value: {initialValue:.8g}")
print(f"Updated value: {float(parameter.getParameterValue()):.8g}")

Initial value: 1
Updated value: 1.1
Flux Systematics/#0
Initial value: 1
Updated value: 1.1


In [8]:
likelihoodInterface.propagateAndEvalLikelihood()
updatedLlh = float(likelihoodInterface.getLastLikelihood())

print(f"Updated LLH: {updatedLlh:.8g}")
print(f"Delta LLH:   {updatedLlh - nominalLlh:.8g}")

Updated LLH: 4043.8009
Delta LLH:   10.325902


In [9]:
parameter.setParameterValue(initialValue)
likelihoodInterface.propagateAndEvalLikelihood()
restoredLlh = float(likelihoodInterface.getLastLikelihood())

print(f"Restored value: {float(parameter.getParameterValue()):.8g}")
print(f"Restored LLH:   {restoredLlh:.8g}")

Restored value: 1
Restored LLH:   4033.475
